In [38]:
#%% Ячейка 0. Установка зависимостей
# ============================================================
import sys, subprocess

def _pip(*args):
    """Ставит пакеты в то же окружение Python, где запущен ноутбук."""
    
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args])

# pymorphy3 — морфологический анализатор (форк pymorphy2 с обновлёнными словарями);
# pymorphy3-dicts-ru — русские словари для него (отдельный пакет, чтобы не тянуть все языки);
# faker — генератор случайных имён и фамилий с локалью ru_RU.
_pip("pymorphy3", "pymorphy3-dicts-ru", "faker")

print("✅ Установка завершена. Перезапустите ядро (Kernel → Restart).")

✅ Установка завершена. Перезапустите ядро (Kernel → Restart).


In [39]:
#%% Ячейка 1. Импорты и конфигурация
# ============================================================
import re
import random
from typing import List, Tuple, Dict, Optional

import pymorphy3
from faker import Faker

# ---- Морфологический анализатор и генератор ----
# MorphAnalyzer — «движок» pymorphy3: умеет разбирать слово на граммемы
# (род, число, падеж) и приводить его к нужной форме (склонять).
morph = pymorphy3.MorphAnalyzer()

# Faker с локалью ru_RU — генерирует имена и фамилии, типичные для России.
fake = Faker("ru_RU")

# SystemRandom — источник энтропии от ОС (urandom), не зависит от seed.
# Нужен для криптостойкой случайности: повторный вызов anon должен давать
# новые значения, и их нельзя предсказать по предыдущим.
rng = random.SystemRandom()

# ---- Поддерживаемые атрибуты ----
# Совпадают с тем, что умеет распознавать ЛР1 (КМ2, чётный вариант — минус
# OGRNIP/IP/BIRTH_CERT/KLADR, которые по заданию ЛР2 не обезличиваются).
SUPPORTED_ATTRS = ("SURNAME", "NAME", "EMAIL", "INN", "PHONE", "CREDIT_CARD")

# ---- Валидаторы контрольных разрядов (для самопроверки после замены) ----
def validate_inn(digits: str) -> bool:
    """Проверяет 12-значный ИНН по алгоритму ФНС."""
    d = re.sub(r"\D", "", digits)
    if len(d) != 12:
        return False
    nums = [int(c) for c in d]
    c11 = [7, 2, 4, 10, 3, 5, 9, 4, 6, 8]
    c12 = [3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8]
    n11 = (sum(nums[i] * c11[i] for i in range(10)) % 11) % 10
    if n11 != nums[10]:
        return False
    n12 = (sum(nums[i] * c12[i] for i in range(11)) % 11) % 10
    return n12 == nums[11]


def validate_luhn(card_number: str) -> bool:
    """Проверяет 16-значный номер карты по алгоритму Луна."""
    d = re.sub(r"\D", "", card_number)
    if len(d) != 16:
        return False
    total = 0
    for i, ch in enumerate(reversed(d)):
        n = int(ch)
        if i % 2 == 1:
            n *= 2
            if n > 9:
                n -= 9
        total += n
    return total % 10 == 0


# Быстрые проверки — если логика сломается, упадёт сразу
assert validate_inn("770698959438") is True
assert validate_luhn("4561261212345467") is True
print("Конфигурация загружена. Поддерживаемых атрибутов:", len(SUPPORTED_ATTRS))

Конфигурация загружена. Поддерживаемых атрибутов: 6


In [40]:
#%% Ячейка 2. Морфологические функции (с защитой от неоднозначных имён)
# ============================================================
# pymorphy3 разбирает слово в список возможных интерпретаций (Parse).
# Для ФИО это принципиально: «Стоян» может разобраться как что угодно,
# кроме имени (в словаре его нет). Если склонять такой разбор, получаются
# прилагательные вроде «стоянный». Поэтому генерируем имя/фамилию и
# проверяем, что pymorphy видит её как Name/Surn — иначе пробуем ещё раз.

GRAMMEME_KEYS = ("gender", "number", "case")


def _find_parse_with_tag(word: str, tag_name: str):
    """Возвращает первый разбор pymorphy, в теге которого есть tag_name.
    
    Для имён это "Name", для фамилий "Surn", для отчеств "Patr".
    Если ни один разбор не содержит нужного тега — возвращает None
    (слово не является именем/фамилией в словаре pymorphy).
    """
    for p in morph.parse(word):
        if tag_name in p.tag:
            return p
    return None


def get_grammemes(word: str) -> Dict[str, Optional[str]]:
    """Возвращает род, число и падеж слова.
    
    Приоритет отдаётся разбору как имени/фамилии/отчеству — если такой
    есть, потому что разбор как прилагательного может дать неверный
    падеж (у неодушевлённых прилагательных мужского рода именительный
    и винительный совпадают). Если специального разбора нет — берём
    первый попавшийся (для не-ФИО этого достаточно).
    """
    parses = morph.parse(word)
    if not parses:
        return {"gender": "masc", "number": "sing", "case": "nomn"}
    # Ищем «человеческий» разбор
    chosen = parses[0]
    for p in parses:
        if any(t in p.tag for t in ("Name", "Surn", "Patr")):
            chosen = p
            break
    tag = chosen.tag
    return {
        "gender": tag.gender or "masc",
        "number": tag.number or "sing",
        "case":   tag.case   or "nomn",
    }


def inflect_to(word: str, grammemes: Dict[str, Optional[str]]) -> str:
    """Приводит слово к заданным граммемам (число, падеж).
    
    Род не входит в целевую форму — он фиксирован самим словом.
    Если исходное слово не склоняется (неизменяемое имя вроде «Джон»),
    inflect() может вернуть None или исходную форму — обрабатываем это
    на уровне вызывающего кода (generate_*_with_morphology).
    """
    target = {g for g in (grammemes.get("case"), grammemes.get("number")) if g}
    if not target:
        return word
    parses = morph.parse(word)
    if not parses:
        return word
    inflected = parses[0].inflect(target)
    return inflected.word if inflected else word


def _generate_inflected(original: str, tag_name: str) -> str:
    """Универсальная генерация: имя или фамилия с сохранением граммем.
    
    tag_name: "Name" для имён, "Surn" для фамилий.
    
    Логика:
    1. Определяем целевые граммемы по оригиналу (get_grammemes).
    2. Пробуем до 200 раз: генерируем случайное слово через Faker,
       проверяем, что оно разбирается как Name/Surn (иначе — пропускаем),
       склоняем к целевой форме, снова проверяем разбор.
    3. Если за 200 попыток валидного результата не нашлось (крайне
       маловероятно), возвращаем оригинал без изменений.
    """
    gr = get_grammemes(original)
    target = {g for g in (gr["case"], gr["number"]) if g}
    for _ in range(200):
        if tag_name == "Name":
            raw = fake.first_name_female() if gr["gender"] == "femn" else fake.first_name_male()
        else:
            raw = fake.last_name_female() if gr["gender"] == "femn" else fake.last_name_male()

        # 1) Слово из Faker должно быть именем/фамилией по мнению pymorphy
        parse = _find_parse_with_tag(raw, tag_name)
        if parse is None:
            continue
        # 2) Faker отдаёт слова в именительном падеже — убеждаемся в этом
        if parse.tag.case not in (None, "nomn"):
            continue
        # 3) Склоняем к целевой форме
        inflected = parse.inflect(target)
        if inflected is None:
            continue
        # 4) Результат должен снова разбираться как Name/Surn
        result_parse = _find_parse_with_tag(inflected.word, tag_name)
        if result_parse is None:
            continue
        # 5) И падеж результата должен совпасть с целевым
        if gr["case"] and result_parse.tag.case != gr["case"]:
            continue
        return inflected.word
    # Fallback: все 200 попыток неудачны — возвращаем оригинал
    return original


def generate_name_with_morphology(original: str) -> str:
    """Генерирует случайное имя, сохраняя род, число и падеж оригинала.
    
    Пример: «Ивана» (м.р., ед.ч., род.п.) → «Петра» (м.р., ед.ч., род.п.).
    """
    return _generate_inflected(original, "Name")


def generate_surname_with_morphology(original: str) -> str:
    """Генерирует случайную фамилию, сохраняя род, число и падеж оригинала.
    
    Пример: «Ивановой» (ж.р., ед.ч., род.п.) → «Петровой» (ж.р., ед.ч., род.п.).
    """
    return _generate_inflected(original, "Surn")


# Смоук-тест: проверяем, что все разборы — ожидаемого типа
for w in ["Иван", "Ивана", "Ивану", "Иванова", "Ивановой", "Петров", "Петрова"]:
    gr = get_grammemes(w)
    parse_name = _find_parse_with_tag(w, "Name")
    parse_surn = _find_parse_with_tag(w, "Surn")
    kind = "Name" if parse_name else ("Surn" if parse_surn else "?")
    print(f"  {w:12s} → {kind:5s} {gr}")

  Иван         → Name  {'gender': 'masc', 'number': 'sing', 'case': 'nomn'}
  Ивана        → Name  {'gender': 'masc', 'number': 'sing', 'case': 'gent'}
  Ивану        → Name  {'gender': 'masc', 'number': 'sing', 'case': 'datv'}
  Иванова      → Surn  {'gender': 'masc', 'number': 'sing', 'case': 'gent'}
  Ивановой     → Surn  {'gender': 'femn', 'number': 'sing', 'case': 'gent'}
  Петров       → Name  {'gender': 'masc', 'number': 'sing', 'case': 'nomn'}
  Петрова      → Surn  {'gender': 'masc', 'number': 'sing', 'case': 'gent'}


In [41]:
#%% Ячейка 3. Обезличивание структурированных атрибутов
# ============================================================

def obfuscate_email(email: str) -> str:
    """Заменяет локальную часть e-mail на случайную строку.
    
    Домен после «@» сохраняется — это косвенный признак (например, корпоративный
    домен может быть значим для аналитики). Если строка не похожа на e-mail
    (нет «@» или локальная часть пуста) — возвращается без изменений.
    """
    if not isinstance(email, str) or "@" not in email:
        return email
    local, domain = email.split("@", 1)
    if not local or not domain:
        return email
    alphabet = "abcdefghijklmnopqrstuvwxyz0123456789"
    # Длину сохраняем — чтобы не ломать визуальное форматирование
    new_local = "".join(rng.choice(alphabet) for _ in range(max(len(local), 4)))
    return f"{new_local}@{domain}"


def obfuscate_inn(inn: str) -> str:
    """Обезличивает ИНН: сохраняет первые 4 цифры и валидные контрольные разряды.
    
    Первые 4 цифры — код налогового органа (косвенный признак региона).
    Остальные 6 цифр (позиции 5–10) генерируются случайно, а 11-я и 12-я
    пересчитываются по алгоритму ФНС, чтобы номер оставался валидным.
    Разделители (пробелы, дефисы) сохраняются на своих местах.
    """
    if not isinstance(inn, str):
        return inn
    digits = re.sub(r"\D", "", inn)
    if len(digits) != 12:
        return inn
    prefix = digits[:4]
    middle = "".join(str(rng.randint(0, 9)) for _ in range(6))
    nums = [int(c) for c in (prefix + middle)]
    c11 = [7, 2, 4, 10, 3, 5, 9, 4, 6, 8]
    c12 = [3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8]
    n11 = (sum(nums[i] * c11[i] for i in range(10)) % 11) % 10
    nums11 = nums + [n11]
    n12 = (sum(nums11[i] * c12[i] for i in range(11)) % 11) % 10
    new_digits = prefix + middle + str(n11) + str(n12)
    # Возвращаем цифры на те же позиции, что были в исходной строке
    it = iter(new_digits)
    return "".join(next(it) if c.isdigit() else c for c in inn)


def obfuscate_phone(phone: str) -> str:
    """Обезличивает номер телефона: сохраняет первые 4 цифры и нецифровые символы.
    
    Первые 4 цифры — префикс + код оператора (например, «+7 (999)» → 7999).
    Все скобки, дефисы, пробелы, плюсы остаются на своих местах — чтобы
    формат был визуально тем же. Остальные цифры генерируются случайно.
    """
    if not isinstance(phone, str):
        return phone
    digit_positions = [i for i, c in enumerate(phone) if c.isdigit()]
    if len(digit_positions) < 5:
        return phone
    # Индексы первых 4 цифр — их сохраняем
    fixed = set(digit_positions[:4])
    result = list(phone)
    for pos in digit_positions:
        if pos not in fixed:
            result[pos] = str(rng.randint(0, 9))
    return "".join(result)


def obfuscate_credit_card(card: str) -> str:
    """Обезличивает номер карты: сохраняет первые 6 и последние 4 цифры.
    
    Первые 6 — BIN (банк-эмитент), последние 4 обычно показываются
    пользователю в интерфейсах. Средние 6 цифр (позиции 7–12) генерируются.
    Контрольная цифра (16-я) фиксирована — поэтому 12-ю цифру подбираем
    перебором так, чтобы номер проходил алгоритм Луна.
    Разделители (пробелы, дефисы) сохраняются.
    """
    if not isinstance(card, str):
        return card
    digits = re.sub(r"\D", "", card)
    if len(digits) != 16:
        return card
    prefix = digits[:6]                  # d1..d6
    suffix = digits[12:16]               # d13..d16, d16 — контрольная
    rng_free = "".join(str(rng.randint(0, 9)) for _ in range(5))  # d7..d11
    new_digits = None
    # Перебираем d12 от 0 до 9 — ровно один вариант сделает сумму Луна кратной 10
    for d12 in range(10):
        candidate = prefix + rng_free + str(d12) + suffix
        if validate_luhn(candidate):
            new_digits = candidate
            break
    if new_digits is None:
        return card
    it = iter(new_digits)
    return "".join(next(it) if c.isdigit() else c for c in card)

In [42]:
#%% Ячейка 4. Главная функция anon
# ============================================================

def _obfuscate_value(attr: str, value: str) -> str:
    """Диспетчеризация: выбирает функцию обезличивания по типу атрибута.
    
    Если тип не поддерживается — возвращает значение без изменений.
    Все исключения внутри конкретных функций перехватываются на уровне anon,
    поэтому здесь достаточно простого соответствия.
    """
    dispatch = {
        "SURNAME":     generate_surname_with_morphology,
        "NAME":        generate_name_with_morphology,
        "EMAIL":       obfuscate_email,
        "INN":         obfuscate_inn,
        "PHONE":       obfuscate_phone,
        "CREDIT_CARD": obfuscate_credit_card,
    }
    func = dispatch.get(attr)
    return func(value) if func else value


def anon(text: str, entities: List[Tuple[str, int, int]]) -> str:
    """Обезличивает ПДн в тексте по списку сущностей.
    
    Параметры:
        text     — исходный текст;
        entities — список (ATTR, start, end) в символьных индексах текста.
    
    Возвращает текст с заменами. Крайние случаи:
        * пустой текст или пустой список — возвращает текст как есть;
        * некорректные индексы (start < 0, end > len, start >= end) — сущность
          пропускается;
        * перекрывающиеся сущности — оставляется только первая (по start),
          последующие отбрасываются;
        * неизвестный тип атрибута — значение остаётся без замены.
    
    Замены применяются справа налево (через сортировку по start по убыванию),
    чтобы индексы предыдущих сущностей не сдвигались после вставки более
    длинных новых значений. Однако для читаемости проще собирать результат
    через срезы от начала — это безопасно, так как все исходные индексы
    относятся к оригинальному тексту, а не к промежуточному результату.
    """
    if not text or not entities:
        return text
    
    # 1. Фильтруем некорректные и перекрывающиеся сущности
    valid = []
    for e in entities:
        if not (isinstance(e, (tuple, list)) and len(e) == 3):
            continue
        attr, start, end = e
        if not (isinstance(attr, str) and isinstance(start, int) and isinstance(end, int)):
            continue
        if start < 0 or end > len(text) or start >= end:
            continue
        valid.append((attr, start, end))
    valid.sort(key=lambda x: x[1])
    
    # Удаляем перекрытия: сохраняем более раннюю, отбрасываем всё, что
    # начинается до конца уже принятой
    filtered = []
    last_end = 0
    for attr, start, end in valid:
        if start < last_end:
            continue
        filtered.append((attr, start, end))
        last_end = end
    
    # 2. Собираем результат через срезы (индексы — от оригинального текста)
    parts = []
    prev = 0
    for attr, start, end in filtered:
        parts.append(text[prev:start])
        original = text[start:end]
        try:
            replaced = _obfuscate_value(attr, original)
        except Exception as exc:
            # Любая проблема с конкретной заменой не должна ломать весь текст
            print(f"⚠ Ошибка при обезличивании {attr}='{original}': {exc}")
            replaced = original
        parts.append(replaced)
        prev = end
    parts.append(text[prev:])
    return "".join(parts)

In [43]:
#%% Ячейка 5. Демонстрация
# ============================================================

demo_text = "Иванов Иван, почта ivan.petrov@example.ru, ИНН 770698959438, тел. +7 (999) 777-12-34, карта 4561 2612 1234 5467."
demo_entities = [
    ("SURNAME",     0, 6),      # Иванов
    ("NAME",        7, 11),     # Иван
    ("EMAIL",       20, 42),    # ivan.petrov@example.ru
    ("INN",         49, 61),    # 770698959438
    ("PHONE",       68, 87),    # +7 (999) 777-12-34
    ("CREDIT_CARD", 94, 113),   # 4561 2612 1234 5467
]

print("=== Демонстрация обезличивания ===")
print("Исходный:  ", demo_text)
print("Обезличен: ", anon(demo_text, demo_entities))

print("\n=== Проверка случайности (два вызова подряд) ===")
for i in range(2):
    print(f"Попытка {i+1}: {anon(demo_text, demo_entities)}")

print("\n=== Морфология: разные падежи и роды ===")
morph_demos = [
    ("Иван",     [("NAME", 0, 4)]),
    ("Ивана",    [("NAME", 0, 5)]),
    ("Ивану",    [("NAME", 0, 5)]),
    ("Ивановой", [("SURNAME", 0, 8)]),
    ("Петрову",  [("SURNAME", 0, 7)]),
]
for text, ents in morph_demos:
    print(f"  {text:12s} → {anon(text, ents)}")

=== Демонстрация обезличивания ===
Исходный:   Иванов Иван, почта ivan.petrov@example.ru, ИНН 770698959438, тел. +7 (999) 777-12-34, карта 4561 2612 1234 5467.
Обезличен:  данилов мартьян, почта iqq3c7j09r1@example.ru, ИНН 770698959438, тел. +7 (999) 701-99-74, карта 4561 2612 1234 5467.

=== Проверка случайности (два вызова подряд) ===
Попытка 1: суворов самуил, почта iid8kczbdfx@example.ru, ИНН 770698959438, тел. +7 (999) 708-77-15, карта 4561 2612 1234 5467.
Попытка 2: терентьев елизар, почта ix6ujqt9puz@example.ru, ИНН 770698959438, тел. +7 (999) 721-68-18, карта 4561 2612 1234 5467.

=== Морфология: разные падежи и роды ===
  Иван         → касьян
  Ивана        → ивана
  Ивану        → игнатию
  Ивановой     → марковой
  Петрову      → доронину


In [44]:
#%% Ячейка 6. Тесты и проверки
# ============================================================
import re as _re

# ---- 1. Email: домен сохраняется ----
for _ in range(20):
    src = "user.name@corp.example.ru"
    out = obfuscate_email(src)
    assert "@corp.example.ru" in out, f"Домен потерян: {out}"
    assert out != src, "Email не изменился"
print("✅ EMAIL: домен сохранён, локальная часть заменена")

# ---- 2. ИНН: префикс сохранён, контроль валиден ----
for _ in range(20):
    src_inn = "770698959438"
    out_inn = obfuscate_inn(src_inn)
    digits = _re.sub(r"\D", "", out_inn)
    assert digits[:4] == src_inn[:4], "Префикс ИНН изменился"
    assert validate_inn(digits), f"ИНН невалиден: {out_inn}"
    assert digits != src_inn, "ИНН не изменился"
print("✅ INN: префикс сохранён, контрольные разряды валидны")

# ---- 3. PHONE: первые 4 цифры и нецифровые символы сохранены ----
for _ in range(20):
    src_phone = "8999 777-12-34"
    out_phone = obfuscate_phone(src_phone)
    src_digits = _re.findall(r"\d", src_phone)
    out_digits = _re.findall(r"\d", out_phone)
    assert src_digits[:4] == out_digits[:4], "Первые 4 цифры телефона изменились"
    # Нецифровые символы на своих местах
    src_syms = [c for c in src_phone if not c.isdigit()]
    out_syms = [c for c in out_phone if not c.isdigit()]
    assert src_syms == out_syms, "Разделители телефона изменились"
    assert src_phone != out_phone, "Телефон не изменился"
print("✅ PHONE: первые 4 цифры и нецифровые символы сохранены")

# ---- 4. CREDIT_CARD: префикс и суффикс сохранены, Луна валиден ----
for _ in range(20):
    src_card = "4561 2612 1234 5467"
    out_card = obfuscate_credit_card(src_card)
    src_digits = _re.sub(r"\D", "", src_card)
    out_digits = _re.sub(r"\D", "", out_card)
    assert out_digits[:6] == src_digits[:6], "Первые 6 цифр карты изменились"
    assert out_digits[-4:] == src_digits[-4:], "Последние 4 цифры карты изменились"
    assert validate_luhn(out_digits), f"Карта невалидна: {out_card}"
    assert out_digits != src_digits, "Карта не изменилась"
print("✅ CREDIT_CARD: BIN и суффикс сохранены, Луна валиден")

# ---- 5. Морфология: род и падеж сохраняются ----
cases = [
    ("Иван",     "masc", "nomn"),
    ("Ивана",    "masc", "gent"),
    ("Ивану",    "masc", "datv"),
    ("Мария",    "femn", "nomn"),
    ("Марию",    "femn", "accs"),
    ("Петровой", "femn", "gent"),
]
for word, expected_gender, expected_case in cases:
    for _ in range(10):
        new_word = generate_name_with_morphology(word) if word.lower() in ("иван", "ивана", "ивану", "мария", "марию") else generate_surname_with_morphology(word)
        gr = get_grammemes(new_word)
        assert gr["gender"] == expected_gender, f"{word} → {new_word}: род {gr['gender']} != {expected_gender}"
        assert gr["case"] == expected_case, f"{word} → {new_word}: падеж {gr['case']} != {expected_case}"
print("✅ Морфология: род и падеж сохраняются")

# ---- 6. Случайность: повторный вызов даёт разные значения ----
results = {anon("Иван", [("NAME", 0, 4)]) for _ in range(50)}
assert len(results) > 1, "Все замены одинаковые — нет случайности"
print(f"✅ Случайность: {len(results)} уникальных замен имени из 50")

# ---- 7. Крайние случаи: не падает на пустом / мусоре ----
assert anon("", []) == ""
assert anon("просто текст", []) == "просто текст"
assert anon("", [("NAME", 0, 4)]) == ""
# Некорректные индексы
assert anon("abc", [("NAME", 5, 10)]) == "abc"
assert anon("abc", [("NAME", -1, 2)]) == "abc"
# Неизвестный тип атрибута — значение остаётся
assert anon("abc", [("UNKNOWN", 0, 3)]) == "abc"
# Невалидный email — не падаем
assert obfuscate_email("no-at-sign") == "no-at-sign"
assert obfuscate_email("") == ""
# Невалидный ИНН (11 цифр) — без изменений
assert obfuscate_inn("77069895943") == "77069895943"
print("✅ Крайние случаи: пустой текст, мусор, неизвестные атрибуты — без падений")

print("\n🎉 Все проверки пройдены.")

✅ EMAIL: домен сохранён, локальная часть заменена
✅ INN: префикс сохранён, контрольные разряды валидны
✅ PHONE: первые 4 цифры и нецифровые символы сохранены
✅ CREDIT_CARD: BIN и суффикс сохранены, Луна валиден
✅ Морфология: род и падеж сохраняются
✅ Случайность: 49 уникальных замен имени из 50
✅ Крайние случаи: пустой текст, мусор, неизвестные атрибуты — без падений

🎉 Все проверки пройдены.


In [45]:
# %pip install nbformat

In [46]:
# %run Recognition_PDn_KR2.ipynb

In [47]:
#%% Ячейка N. Recognizer из КМ2 — самодостаточная копия
# ============================================================
# Загружает обученную модель из ./rubert_pii, реализует правила для
# структурированных атрибутов, объединяет BERT+правила в predict().
# Никаких импортов из других ноутбуков — всё в одном месте.

import os, re
import torch
from transformers import pipeline as hf_pipeline, AutoTokenizer
from typing import List, Tuple

MODEL_OUT = "./rubert_pii"
MAX_LEN = 256

# ---- загрузка обученной модели ----
try:
    ner_pipe = hf_pipeline(
        "token-classification", model=MODEL_OUT, tokenizer=MODEL_OUT,
        aggregation_strategy="simple",
        device=0 if torch.cuda.is_available() else -1,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_OUT)
    MODEL_READY = True
except Exception as e:
    print("⚠ Модель не загружена, работаем только по правилам:", e)
    ner_pipe = None
    tokenizer = None
    MODEL_READY = False

# ---- валидаторы контрольных разрядов ----
def _digits_only(s): return re.sub(r"\D", "", s)

def validate_inn(d):
    d = _digits_only(d)
    if len(d) != 12: return False
    n = [int(c) for c in d]
    c11 = [7,2,4,10,3,5,9,4,6,8]; c12 = [3,7,2,4,10,3,5,9,4,6,8]
    n11 = (sum(n[i]*c11[i] for i in range(10)) % 11) % 10
    if n11 != n[10]: return False
    n12 = (sum(n[i]*c12[i] for i in range(11)) % 11) % 10
    return n12 == n[11]

def validate_luhn(d):
    d = _digits_only(d)
    if len(d) != 16: return False
    total = 0
    for i, ch in enumerate(reversed(d)):
        v = int(ch)
        if i % 2 == 1:
            v *= 2
            if v > 9: v -= 9
        total += v
    return total % 10 == 0

PASSPORT_REGION_WHITELIST = {
    1,3,4,5,7,8,10,11,12,14,15,17,18,19,20,21,22,24,25,26,27,28,29,
    32,33,34,35,36,37,38,40,41,42,43,44,45,46,47,48,49,50,51,52,53,
    54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,
    75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,
    96,97,98,99,
}

def validate_passport_series(s):
    s = _digits_only(s)
    return len(s) == 4 and int(s[:2]) in PASSPORT_REGION_WHITELIST

def validate_ogrnip(d):
    d = _digits_only(d)
    return len(d) == 15 and (int(d[:14]) % 13) % 10 == int(d[14])

def validate_kladr(d):
    d = _digits_only(d)
    return len(d) in (13, 17) and int(d[:2]) in PASSPORT_REGION_WHITELIST

# ---- регулярки ----
EMAIL_RE = re.compile(r"(?i)([a-z0-9а-яё_.+\-]+@[a-zа-яё\d][a-zа-яё\d.\-]*\.[a-zа-яё]{2,})")
IP_RE = re.compile(r"(?<![\d.])(?:(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)\.){3}"
                   r"(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)(?!\.?\d)")
INN_RE = re.compile(r"(?<!\d)(\d[\s\-]?){11}\d(?!\d)")
CARD_RE = re.compile(r"(?<!\d)(?:\d[\s\-]?){15}\d(?!\d)")
PASSPORT_RE = re.compile(r"(?<!\d)(\d{2})[\s\-]?(\d{2})[\s\-]?(\d{6})(?!\d)")
OGRNIP_RE = re.compile(r"(?<!\d)(\d[\s\-]?){14}\d(?!\d)")
KLADR_RE = re.compile(r"(?<!\d)(\d{13}|\d{17})(?!\d)")
BIRTH_CERT_RE = re.compile(r"(?<![А-ЯЁA-Z\d])([IVX]{1,4})[\s\-]?(?:[А-ЯЁA-Z]{2}[\s\-]?)?(\d{6,7})(?!\d)")

def extract_by_rules(text):
    """Прогоняет 8 регулярок + проверки контрольных разрядов."""
    if not text: return []
    res = []
    res += [("EMAIL", m.start(), m.end()) for m in EMAIL_RE.finditer(text)]
    res += [("IP", m.start(), m.end()) for m in IP_RE.finditer(text)]
    for m in INN_RE.finditer(text):
        if validate_inn(m.group(0)): res.append(("INN", m.start(), m.end()))
    for m in CARD_RE.finditer(text):
        if validate_luhn(m.group(0)): res.append(("CREDIT_CARD", m.start(), m.end()))
    for m in PASSPORT_RE.finditer(text):
        if validate_passport_series(m.group(1) + m.group(2)):
            res.append(("PASSPORT", m.start(), m.end()))
    for m in OGRNIP_RE.finditer(text):
        if validate_ogrnip(m.group(0)): res.append(("OGRNIP", m.start(), m.end()))
    for m in KLADR_RE.finditer(text):
        if validate_kladr(m.group(0)): res.append(("KLADR", m.start(), m.end()))
    res += [("BIRTH_CERT", m.start(), m.end()) for m in BIRTH_CERT_RE.finditer(text)]
    return res

# ---- BERT с чанкингом ----
def model_entities(text):
    """BERT + чанкинг: текст режется на окна по MAX_LEN с перекрытием 50%."""
    if not text or ner_pipe is None: return []
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    offsets = enc["offset_mapping"]
    if not offsets: return []
    chunk_size = MAX_LEN - 4
    stride = chunk_size // 2
    out, seen = [], set()
    start_idx = 0
    while start_idx < len(offsets):
        end_idx = min(start_idx + chunk_size, len(offsets))
        cs, ce = offsets[start_idx][0], offsets[end_idx - 1][1]
        chunk_text = text[cs:ce]
        for ent in ner_pipe(chunk_text):
            label = ent["entity_group"]
            if label not in ("NAME", "SURNAME"): continue
            key = (label, cs + int(ent["start"]), cs + int(ent["end"]))
            if key in seen: continue
            seen.add(key)
            out.append((label, key[1], key[2]))
        if end_idx >= len(offsets): break
        start_idx += stride
    return out

def _overlap(a, b):
    return not (a[2] <= b[1] or a[1] >= b[2])

def merge_entities(rule_ents, model_ents):
    """Правило > модель при конфликте; при одинаковом типе — длиннее."""
    all_ents = [(l,s,e,"rule") for l,s,e in rule_ents] + \
               [(l,s,e,"model") for l,s,e in model_ents]
    all_ents.sort(key=lambda x: (0 if x[3]=="rule" else 1, -(x[2]-x[1])))
    result, used = [], []
    for l,s,e,src in all_ents:
        if any(_overlap((l,s,e), u) for u in used):
            same = [u for u in used if u[0]==l and _overlap((l,s,e), u)]
            if same:
                longer = max(same, key=lambda x: x[2]-x[1])
                if (e-s) > (longer[2]-longer[1]):
                    used = [u for u in used if u != longer]
                    result = [r for r in result if r != longer]
                    result.append((l,s,e)); used.append((l,s,e))
            continue
        result.append((l,s,e)); used.append((l,s,e))
    return sorted(result, key=lambda x: x[1])

def predict(text):
    """Финальная функция распознавания ПДн."""
    if not text or not text.strip(): return []
    return merge_entities(extract_by_rules(text), model_entities(text))

print("✅ Recognizer готов:", "BERT+правила" if MODEL_READY else "только правила")
print("   Модель:", MODEL_OUT)

✅ Recognizer готов: BERT+правила
   Модель: ./rubert_pii


In [48]:
#%% Ячейка N+1. Проверка recognizer
# ============================================================
test = "Иванов Иван, почта ivan@mail.ru, ИНН 770698959438."
print("Распознано:", predict(test))
# Ожидаемо: [('SURNAME', 0, 6), ('NAME', 7, 11), ('EMAIL', 19, 31), ('INN', 37, 49)]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Распознано: [('SURNAME', 0, 6), ('NAME', 7, 11), ('EMAIL', 19, 31), ('INN', 37, 49)]
